In [3]:
%pip install snowpark_batch

Note: you may need to restart the kernel to use updated packages.


In [2]:
import random
import pandas as pd

random.seed(42)

base_companies = [
    "Acme Technologies", "Global Foods", "Blue Ocean Logistics", "NextGen Systems",
    "Café Central", "Niño Retail", "München Motors", "São Energy", "Crème Labs",
    "Zoë Health", "İstanbul Textiles", "Résumé Media", "North Star Finance",
    "Green Leaf Organics", "Rapid Build Infra", "Urban Mobility", "Prime Legal",
    "Silverline Pharma", "Skyline Aviation", "Pioneer Data", "Quantum Security",
    "Everbright Electronics", "Atlas Mining", "Aurora Biotech", "Vertex Telecom"
]

suffixes = ["LLC", "Inc", "Ltd", "Corp", "GmbH", "S.A.", "Pvt Ltd"]
symbols = [" & ", " + ", " @ "]

def random_case(text: str) -> str:
    style = random.choice(["lower", "upper", "title", "mixed"])
    if style == "lower":
        return text.lower()
    if style == "upper":
        return text.upper()
    if style == "title":
        return text.title()
    # mixed case
    return "".join(c.upper() if random.random() > 0.5 else c.lower() for c in text)

def noisy_variant(base: str) -> str:
    name = base

    # optionally add a legal suffix
    if random.random() < 0.75:
        name = f"{name} {random.choice(suffixes)}"

    # optionally inject symbol between first two words
    parts = name.split()
    if len(parts) >= 2 and random.random() < 0.45:
        parts = [parts[0], random.choice(symbols).strip(), " ".join(parts[1:])]
        name = " ".join(parts)

    # random case changes
    name = random_case(name)

    # inject double spaces
    if random.random() < 0.6:
        name = name.replace(" ", "  ", random.randint(1, min(3, name.count(" "))))

    return name

# Build 100 records (multiple variants of base companies -> different group sizes after normalization)
records = []
for i in range(1, 101):
    base = random.choice(base_companies)
    records.append({"id": i, "company_name": noisy_variant(base)})

df = pd.DataFrame(records)
df.head(10)

,id,company_name
0,1,QUANTUM & SECURITY S.A.
1,2,everbright + electronics gmbh
2,3,sÃO ENerGy lLC
3,4,auROra BIOTEch llC
4,5,EVERBRIGHT ELECTRONICS GMBH
5,6,CRÈME LABS
6,7,Vertex & Telecom
7,8,München Motors
8,9,NORTH STAR FINANCE
9,10,AurORa bIOtecH LTd


In [26]:
from snowpark_batch import batch_dataframe

result = batch_dataframe(
    df,
    key_column="company_name",
    estimated_batch_size=20,
    lowercase=True,
    remove_suffixes=False,
    transliterate_unicode=True,
    replace_symbols=False,
)
# result has original columns + company_name_norm + batch_id + row_id


In [27]:
result.head(30)

,id,company_name,company_name_norm,row_id,batch_id
0,36,acme + technologies ltd,acme + technologies ltd,1,1
1,15,Acme Technologies Ltd,acme technologies ltd,2,1
2,35,ACMe tEChNolOgies LtD,acme technologies ltd,3,1
3,4,auROra BIOTEch llC,aurora biotech llc,4,1
4,10,AurORa bIOtecH LTd,aurora biotech ltd,5,1
5,69,AtLAs miNiNG s.a.,atlas mining s.a.,6,1
6,31,BLUE & oCEan LoGIsTicS,blue & ocean logistics,7,1
7,21,BLUE OCEAN LOGISTICS,blue ocean logistics,8,1
8,67,BLUE OCEAN LOGISTICS CORP,blue ocean logistics corp,9,1
9,51,bLuE oCEaN lOGISTIcs lTD,blue ocean logistics ltd,10,1
